Le but du challenge est de prédire le genre d'un film à partir de son affiche.
On prends les affiches de film à partir du fichier movies.csv

In [33]:
import csv

with open("movies.csv", "r") as f:
    data = csv.reader(f)
    rows = list(data)

print(rows[:1])
print(len(rows))

[['Release_Date', 'Title', 'Overview', 'Popularity', 'Vote_Count', 'Vote_Average', 'Original_Language', 'Genre', 'Poster_Url']]
9838


On voit donc que le fichier est du type\
['Release_Date', 'Title', 'Overview', 'Popularity', 'Vote_Count', 'Vote_Average', 'Original_Language', 'Genre', 'Poster_Url']\
Le Genre du fillm est donc donné, cela permettra de le comparer la prédiction faite.\
Nous allons utiliser Poster_Url pour prédire le Genre.\
Les données d'entrées de notre réseau de neurones seront donc Genre et les sorties Poster_Url.\
En tout, il y a 9838 films dans le jeu de données movies.csv

Voici un premier exemple de film

In [76]:
i_film = 1
print(rows[i_film])
print(rows[i_film][1])#Title = 1
print(rows[i_film][7])#Genre
print(rows[i_film][8])#Poster_Url


['2021-12-15', 'Spider-Man: No Way Home', 'Peter Parker is unmasked and no longer able to separate his normal life from the high-stakes of being a super-hero. When he asks for help from Doctor Strange the stakes become even more dangerous, forcing him to discover what it truly means to be Spider-Man.', '5083.954', '8940', '8.3', 'en', 'Action, Adventure, Science Fiction', 'https://image.tmdb.org/t/p/original/1g0dhYtq4irTY1GPXvft6k4YLjm.jpg']
Spider-Man: No Way Home
Action, Adventure, Science Fiction
https://image.tmdb.org/t/p/original/1g0dhYtq4irTY1GPXvft6k4YLjm.jpg


Il s'agit du film Spider-Man: No Way Home, qui appartient aux genres Action, Adventure, Science Fiction.Genre dont l'affiche est\
<img src="https://image.tmdb.org/t/p/original/1g0dhYtq4irTY1GPXvft6k4YLjm.jpg" width="200">

Afin de s'assurer que chaque item dans la liste contienne bien un genre et une image, il faut nettoyer les données.\
Les lignes mises en cause sont les suivantes.

In [117]:
#On liste les films qui n'ont pas le bon nombre de colonnes (8), donc a priori pas de Genre
for i, row in enumerate(rows):
    if len(row) < 8:
        print(i, len(row), row)

1106 3 ['2013-10-20', 'Pixie Hollow Bake Off', "Tink challenges Gelata to see who can bake the best cake for the queen's party.  Plus 10 Disney Fairies Mini-Shorts:"]
1107 1 [' - Just Desserts']
1108 1 [' - If The Hue Fits']
1109 1 [' - Dust Up']
1110 1 [' - Scents And Sensibility']
1111 1 [' - Just One Of The Girls']
1112 1 [' - Volleybug']
1113 1 [' - Hide And Tink']
1114 1 [" - Rainbow's Ends"]
1115 1 [' - Fawn And Games']
1116 7 [' - Magic Tricks', '61.328', '35', '7.1', 'en', 'Animation', 'https://image.tmdb.org/t/p/original/6iXYe7AkQ1QIfMFuvXsSCT2zF7s.jpg']


On nettoie donc les données pour n'utiliser que les films qui ont un ou des Genre et une affiche de film.\
À noter que les indices 1106 à 1116 correspondent à une série d'Animation pour lesquels il n'y a qu'une affiche de film mais plusieurs épisodes. Il faut donc les concaténer pour n'en faire qu'un seul film pour lequel on dispose du Genre Animation et d'une affiche correspondante.
<img src="https://image.tmdb.org/t/p/original/6iXYe7AkQ1QIfMFuvXsSCT2zF7s.jpg" width="200">

In [118]:
#['Release_Date', 'Title', 'Overview', 'Popularity', 'Vote_Count', 'Vote_Average', 'Original_Language', 'Genre', 'Poster_Url']
rows.append(['2013-10-20', 'Pixie Hollow Bake Off', "Tink challenges Gelata to see who can bake the best cake for the queen's party.  Plus 10 Disney Fairies Mini-Shorts", '61.328', '35', '7.1', 'en', 'Animation', 'https://image.tmdb.org/t/p/original/6iXYe7AkQ1QIfMFuvXsSCT2zF7s.jpg'])

Et on enlève les lignes 1106 à 1116 pour nettoyer les données.

In [119]:
del rows[1106:1117]

On peut donc lister le Genre de film. Il y en a en tout 19 parmi 9828 films.

In [123]:
len_list = len(rows[:])
print(len_list)
#print(rows[1][7])
#print(rows[len_list-1][7])#-1 Nécessaire pour ne pas dépasser la taille de la liste
genres = set()
for i in range(1, len_list):
    for genre in rows[i][7].split(","):
        genres.add(genre.strip())

print(len(genres))
print(genres)

9828
19
{'Family', 'War', 'Adventure', 'Music', 'Romance', 'Fantasy', 'Documentary', 'Mystery', 'Animation', 'TV Movie', 'Western', 'Thriller', 'Science Fiction', 'Action', 'Crime', 'History', 'Drama', 'Horror', 'Comedy'}


Voici donc la liste des Genre utilisés, les sorties de notre algorithme

In [77]:
print(genres)

{'Family', 'War', 'Adventure', 'Music', 'Romance', 'Fantasy', 'Documentary', 'Mystery', 'Animation', 'TV Movie', 'Western', 'Thriller', 'Science Fiction', 'Action', 'Crime', 'History', 'Drama', 'Genre', 'Horror', 'Comedy'}


Voici des affiches de film qui correspondent à un Genre donné

In [124]:
affiches = []

for row in rows[1:]:  # ignorer l'en-tête
    if len(row) > 8 and "Action" in row[7]:
        affiches.append(row[8])

print(affiches[1:10])

['https://image.tmdb.org/t/p/original/aq4Pwv5Xeuvj6HZKtxyd23e6bE9.jpg', 'https://image.tmdb.org/t/p/original/pSh8MyYu5CmfyWEHzv8FEARH2zq.jpg', 'https://image.tmdb.org/t/p/original/3cccEF9QZgV9bLWyupJO41HSrOV.jpg', 'https://image.tmdb.org/t/p/original/wYihSXWYqN8Ejsdut2P1P0o97N0.jpg', 'https://image.tmdb.org/t/p/original/4NUzcKtYPKkfTwKsLjwNt8nRIXV.jpg', 'https://image.tmdb.org/t/p/original/aw4GGsRwhQtyLsjzC7dsAahfCDY.jpg', 'https://image.tmdb.org/t/p/original/onGdT8sYi89drvSJyEJnft97rOq.jpg', 'https://image.tmdb.org/t/p/original/lAXONuqg41NwUMuzMiFvicDET9Y.jpg', 'https://image.tmdb.org/t/p/original/rjkmN1dniUHVYAtwuV3Tji7FsDO.jpg']


Qui sont les suivantes
<p float="left">
  <img src="https://image.tmdb.org/t/p/original/aq4Pwv5Xeuvj6HZKtxyd23e6bE9.jpg" width="100" />
  <img src="https://image.tmdb.org/t/p/original/pSh8MyYu5CmfyWEHzv8FEARH2zq.jpg" width="100" /> 
  <img src="https://image.tmdb.org/t/p/original/3cccEF9QZgV9bLWyupJO41HSrOV.jpg" width="100" />
</p>

De plus, voici la répartition du nombre d'affiche par Genre.\
Attention, une affiche peut avoir plusieurs Genre.

In [ ]:
compteur_genres = {genre: 0 for genre in genres}

for i in range(1, len_list):
    liste_genres = [g.strip() for g in rows[i][7].split(",")]
    
    for genre in liste_genres:
        compteur_genres[genre] += 1

for genre, nombre in compteur_genres.items():
    print(f"{genre} : {nombre} films")

Family : 1414 films
War : 308 films
Adventure : 1853 films
Music : 295 films
Romance : 1476 films
Fantasy : 1308 films
Documentary : 215 films
Mystery : 773 films
Animation : 1439 films
TV Movie : 214 films
Western : 137 films
Thriller : 2488 films
Science Fiction : 1273 films
Action : 2686 films
Crime : 1242 films
History : 427 films
Drama : 3744 films
Horror : 1470 films
Comedy : 3031 films


ChatGPT recommande de faire une analyse lorsque le cas favorable suivant se présente\
9328 affiches\
15 genres\
Chaque genre présent dans >500 affiches\
2-3 genres par film

À l'instar du cas plus difficile\
9328 affiches\
30 genres\
Certains genres présents dans 50 affiches seulement

Que l'on peut également représenter sous forme d'histogramme.

Voici un graphe représentant la répartitions des genres. Chaque point du graphe représente un film qui peut apparaître comme appartenant à plusieurs genres.

On peut aller plus loin dans l'analyse des données, comme tracer des nuages de points pour les analyses commerciales de films. ref

Pour faire une analyse plus fine, on pourrait essayer d'utiliser les champs lexicaux des titres dans les affiches pour en déduire le Genre.